# Notebook 06 — Phase 1 acceptance

**ATLAS: Aligned Three-Layer Architecture for Semantics**  
FSI Semantic Layer Workshop on AWS — Workshop 2

---

Before I move to Phase 2, how do I know Phase 1 is actually complete?

This notebook runs the full acceptance suite from `spec/10-acceptance-criteria.md`.
Every assertion that passes is a contract honored. Every assertion that fails is
a known issue to resolve before extending to Phase 2.

In [ ]:
# WS2 notebook setup — installs dependencies into THIS kernel.
# uv manages the WS2 virtual environment; this cell installs it
# into the running kernel so imports work without manual setup.
import sys, subprocess, os
from pathlib import Path

# Find the use-case-applications root (3 levels up from phase-1-referral/)
ws2_root = Path(os.getcwd()).parents[1]
venv_python = ws2_root / '.venv' / 'bin' / 'python'

if venv_python.exists() and str(venv_python) != sys.executable:
    # venv exists but we're not running inside it — install into current kernel
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )
elif not venv_python.exists():
    # venv not yet created — run uv sync first
    subprocess.check_call(
        ['uv', 'sync', '--all-groups', '--quiet'],
        cwd=str(ws2_root)
    )
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet',
         '--disable-pip-version-check',
         'rdflib>=7.0.0', 'pyshacl>=0.25.0', 'SPARQLWrapper>=2.0.0',
         'pydantic>=2.0.0'],
        cwd='/tmp'
    )

print('WS2 dependencies ready.')


## The acceptance contract

Phase 1 acceptance is the moment Workshop 2 becomes more than a demo. The
assertions below are organized into seven categories:

1. **Registry completeness** — are all agents and MCP servers registered?
2. **Agent posture compliance** — does each agent honor its declared posture?
3. **Four-layer permission model** — do all four layers compose correctly?
4. **Regulatory compliance** — does the system respect tipping-off and MRM rules?
5. **End-to-end scenario** — does the Rachel Kim referral work start to finish?
6. **GraphQL schema conformance** — does the schema honor the FIBO-shaped contract?
7. **Workshop 1 substrate integrity** — is Workshop 1 untouched?

Categories 1–4 and 6–7 must all pass. Category 5 requires a running Neptune
cluster — if unavailable, those assertions are deferred to deployment.

In [ ]:
import sys
import os
import json
import re

# Workshop 1's shared helpers
sys.path.insert(0, "../../../agentic-semantic-layer/notebooks/shared")

# Spec and source directories
SPEC_DIR = "../../spec/04-aws-agent-registry"
SCHEMA_PATH = "../../spec/05-appsync-graphql/schema.graphql"
ONTOLOGY_EXT_DIR = "../../ontology-extensions"
WORKSHOP_1_DIR = "../../../agentic-semantic-layer"

# Load descriptors
def load_descriptors(subdir):
    path = os.path.join(SPEC_DIR, subdir)
    descriptors = []
    for f in sorted(os.listdir(path)):
        if f.endswith(".json"):
            with open(os.path.join(path, f)) as fh:
                descriptors.append(json.load(fh))
    return descriptors

mcp_descriptors = load_descriptors("mcp-servers")
agent_descriptors = load_descriptors("agents")
phase_1_agents = [d for d in agent_descriptors if d.get("phase") == 1]

# Results tracker
results = {"passed": 0, "failed": 0, "deferred": 0, "details": []}

def check(assertion_id, description, condition, category=""):
    """Record an assertion result."""
    status = "PASS" if condition else "FAIL"
    results["passed" if condition else "failed"] += 1
    results["details"].append({"id": assertion_id, "desc": description, "status": status})
    icon = "✓" if condition else "✗"
    print(f"  {icon} [{assertion_id}] {description}")

def defer(assertion_id, description, reason):
    """Defer an assertion that requires live infrastructure."""
    results["deferred"] += 1
    results["details"].append({"id": assertion_id, "desc": description, "status": "DEFERRED"})
    print(f"  ⊘ [{assertion_id}] {description} — DEFERRED: {reason}")

print("Setup complete. Running acceptance suite...\n")

In [ ]:
# Category 1 — Registry completeness
print("Category 1: Registry completeness")
print("=" * 50)

# 1.1: 5 MCP servers registered
check("1.1", "5 MCP servers are registered", len(mcp_descriptors) == 5)

# 1.2: 5 Phase 1 agents registered
check("1.2", "5 Phase 1 agents are registered", len(phase_1_agents) == 5)

# 1.3: Consumer Banker discovers at least 4 agents
banker_agents = [
    d for d in phase_1_agents
    if "atlas-consumer-banker" in d.get("registry_metadata", {}).get("discoverable_by", [])
]
check("1.3", f"Consumer Banker discovers {len(banker_agents)} agents (need >= 4)",
      len(banker_agents) >= 4)

# 1.4: Wealth Advisor discovers a different set
advisor_agents = [
    d for d in phase_1_agents
    if "atlas-wealth-advisor" in d.get("registry_metadata", {}).get("discoverable_by", [])
]
banker_names = {d["agent_name"] for d in banker_agents}
advisor_names = {d["agent_name"] for d in advisor_agents}
check("1.4", "Wealth Advisor sees different set than Consumer Banker",
      banker_names != advisor_names)

# 1.5: referral-rationale-drafter NOT discoverable by Wealth Advisor
check("1.5", "referral-rationale-drafter NOT discoverable by Wealth Advisor",
      "referral-rationale-drafter" not in advisor_names)

# 1.6: referral-orchestrator discoverable ONLY by Consumer Banker
orch = next(d for d in phase_1_agents if d["agent_name"] == "referral-orchestrator")
orch_discoverable = orch.get("registry_metadata", {}).get("discoverable_by", [])
check("1.6", "referral-orchestrator discoverable only by Consumer Banker",
      orch_discoverable == ["atlas-consumer-banker"])

print()

In [ ]:
# Category 2 — Agent posture compliance
print("Category 2: Agent posture compliance")
print("=" * 50)

# 2.1: nl-to-sparql-agent determinism (structural check — uses embeddings, not text gen)
nl_agent = next(d for d in agent_descriptors if d["agent_name"] == "nl-to-sparql-agent")
check("2.1", "nl-to-sparql-agent posture is deterministic-audited",
      nl_agent["posture"] == "deterministic-audited")

# 2.2: nl-to-sparql-agent uses embedding model, not text generation
env_vars = nl_agent["runtime"]["environment_variables"]
uses_embedding = "BEDROCK_EMBEDDING_MODEL_ID" in env_vars
no_text_gen = "BEDROCK_TEXT_MODEL_ID" not in env_vars
check("2.2", "nl-to-sparql-agent uses embeddings only (no text generation)",
      uses_embedding and no_text_gen)

# 2.3: wealth-signal-detector is SHACL-driven
wsd = next(d for d in agent_descriptors if d["agent_name"] == "wealth-signal-detector")
check("2.3", "wealth-signal-detector depends on atlas-shacl-mcp",
      "atlas-shacl-mcp" in wsd.get("dependencies", {}).get("mcp_servers", []))

# 2.4: referral-rationale-drafter has is_probabilistic in output
rrd = next(d for d in agent_descriptors if d["agent_name"] == "referral-rationale-drafter")
output_required = rrd["output_schema"]["required"]
check("2.4", "referral-rationale-drafter output requires is_probabilistic",
      "is_probabilistic" in output_required)

# 2.5: referral-rationale-drafter has requires_human_review in output
check("2.5", "referral-rationale-drafter output requires requires_human_review",
      "requires_human_review" in output_required)

# 2.6: referral-orchestrator requires approved_rationale
orch_required = orch["input_schema"]["required"]
check("2.6", "referral-orchestrator requires approved_rationale",
      "approved_rationale" in orch_required)

# 2.7: referral-orchestrator restricts persona to consumer-banker
persona_prop = orch["input_schema"]["properties"].get("persona_claim", {})
check("2.7", "referral-orchestrator restricts persona to atlas-consumer-banker",
      persona_prop.get("const") == "atlas-consumer-banker")

print()

In [ ]:
# Category 3 — Four-layer permission model
print("Category 3: Four-layer permission model")
print("=" * 50)

# 3.1: Every MCP server operation requires persona_claim
all_require_persona = True
for desc in mcp_descriptors:
    for op_name, op_schema in desc.get("operations", {}).items():
        inputs = op_schema.get("input", {})
        if inputs and "persona_claim" not in inputs:
            # Some operations (like list_shapes) don't need persona
            if op_name not in ("list_shapes",):
                all_require_persona = False
check("3.1", "Persona claim required on all persona-sensitive MCP operations",
      all_require_persona)

# 3.2: Consumer Banker and Wealth Advisor see different palettes
check("3.2", "Consumer Banker and Wealth Advisor see different capability palettes",
      banker_names != advisor_names)

# 3.3 & 3.4: Require live infrastructure — defer
defer("3.3", "Consumer Banker query returns fewer customers than BSA Analyst",
      "Requires running Neptune cluster with Lake Formation")
defer("3.4", "Consumer Banker cannot traverse BSA-restricted named graphs",
      "Requires running Neptune cluster with named graph scoping")

print()

In [ ]:
# Category 4 — Regulatory compliance
print("Category 4: Regulatory compliance")
print("=" * 50)

# 4.1 & 4.2: Compliance banner tipping-off checks
def render_compliance_banner(has_review, persona):
    if not has_review:
        return None
    if persona == "atlas-bsa-analyst":
        return "SAR draft in progress — BSA team review required"
    return "Active compliance review — contact BSA team before client outreach"

non_bsa_personas = ["atlas-consumer-banker", "atlas-wealth-advisor", "atlas-ontology-steward"]
no_sar_leak = all(
    "SAR" not in render_compliance_banner(True, p) for p in non_bsa_personas
)
no_filed_leak = all(
    "filed" not in render_compliance_banner(True, p).lower() for p in non_bsa_personas
)
check("4.1", "Non-BSA banner does NOT contain 'SAR'", no_sar_leak)
check("4.2", "Non-BSA banner does NOT contain 'filed'", no_filed_leak)

# 4.3: BSA Analyst CAN see SAR detail
bsa_banner = render_compliance_banner(True, "atlas-bsa-analyst")
check("4.3", "BSA Analyst CAN see SAR-specific detail", "SAR" in bsa_banner)

# 4.4: No probabilistic agent auto-commits
prob_agents = [d for d in agent_descriptors if "probabilistic" in d.get("posture", "")]
all_require_review = all(
    "requires_human_review" in d.get("output_schema", {}).get("required", [])
    or "requires_human_review" in d.get("output_schema", {}).get("properties", {})
    for d in prob_agents
)
check("4.4", "All probabilistic agents carry human-review flags", all_require_review)

print()

In [ ]:
# Category 5 — End-to-end Rachel Kim scenario
print("Category 5: End-to-end Rachel Kim scenario")
print("=" * 50)

# All Category 5 assertions require a running Neptune cluster
defer("5.1", "Patel household exists in SLGD", "Requires running Neptune")
defer("5.2", "wealth-signal-detector detects signal for Patel household", "Requires running Neptune")
defer("5.3", "household-traverser returns >= 2 nodes for Patel household", "Requires running Neptune")
defer("5.4", "referral-rationale-drafter produces non-empty draft", "Requires Bedrock access")
defer("5.5", "referral-orchestrator routes successfully", "Requires Step Functions")
defer("5.6", "AuditRecord exists after routing", "Requires running Neptune")
defer("5.7", "Audit record carries PROV-O attribution", "Requires running Neptune")

print()

In [ ]:
# Category 6 — GraphQL schema conformance
print("Category 6: GraphQL schema conformance")
print("=" * 50)

with open(SCHEMA_PATH) as f:
    schema_text = f.read()

# 6.1: Every entity type maps to an ontology class
type_pattern = re.compile(r'"""\n(.+?)\n"""\ntype (\w+)', re.DOTALL)
mapped_types = type_pattern.findall(schema_text)
all_types = re.findall(r'^type (\w+)', schema_text, re.MULTILINE)
infra_types = {"Query", "Mutation", "Subscription", "Provenance", "Capability"}
mapped_names = {name for _, name in mapped_types}
unmapped = set(all_types) - mapped_names - infra_types
check("6.1", f"All entity types map to ontology classes (unmapped: {unmapped or 'none'})",
      len(unmapped) == 0)

# 6.2: Customer type has required fields
has_customer_uri = "uri: ID!" in schema_text
has_customer_id = "customerId: String!" in schema_text
check("6.2", "Customer type has uri and customerId as required fields",
      has_customer_uri and has_customer_id)

# 6.3: WealthSignal has signalType
check("6.3", "WealthSignal type has signalType field",
      "signalType: String!" in schema_text)

# 6.4: Capabilities query exists with personaClaim parameter
check("6.4", "capabilities query accepts personaClaim parameter",
      "capabilities(personaClaim: String!)" in schema_text)

print()

In [ ]:
# Category 7 — Workshop 1 substrate integrity
print("Category 7: Workshop 1 substrate integrity")
print("=" * 50)

# 7.1: No Workshop 1 files modified (check key files exist and are unchanged)
key_w1_files = [
    "ontology/atlas-core.ttl",
    "ontology/atlas-shapes.ttl",
    "ontology/atlas-fibo-alignment.ttl",
    "prompts/ground-truth.yaml",
    "prompts/prefixes.txt",
    "notebooks/shared/atlas_neptune.py",
    "notebooks/shared/atlas_sparql.py",
]
all_exist = all(os.path.exists(os.path.join(WORKSHOP_1_DIR, f)) for f in key_w1_files)
check("7.1", "All key Workshop 1 files exist (not deleted)", all_exist)

# 7.2: 22 ontology classes (count owl:Class declarations in atlas-core + fibo-alignment)
from rdflib import Graph, OWL, RDF
g = Graph()
g.parse(os.path.join(WORKSHOP_1_DIR, "ontology/atlas-core.ttl"), format="turtle")
g.parse(os.path.join(WORKSHOP_1_DIR, "ontology/atlas-fibo-alignment.ttl"), format="turtle")
class_count = len(set(g.subjects(RDF.type, OWL.Class)))
check("7.2", f"Workshop 1 has {class_count} ontology classes (expected 22)",
      class_count == 22)

# 7.3: 6 SHACL shapes exist
shapes_path = os.path.join(WORKSHOP_1_DIR, "ontology/atlas-shapes.ttl")
check("7.3", "Workshop 1 atlas-shapes.ttl exists", os.path.exists(shapes_path))

# 7.4: Workshop 2 extensions use atlas-part-2: namespace
ext_files = [f for f in os.listdir(ONTOLOGY_EXT_DIR) if f.endswith(".ttl")]
all_use_part2 = True
for f in ext_files:
    with open(os.path.join(ONTOLOGY_EXT_DIR, f)) as fh:
        content = fh.read()
    # Check that new classes use atlas-part-2: not atlas:
    if "atlas-part-2:" not in content:
        all_use_part2 = False
check("7.4", "All Workshop 2 extensions use atlas-part-2: namespace", all_use_part2)

print()

In [ ]:
# Final summary
print("\n" + "=" * 60)
print("PHASE 1 ACCEPTANCE SUMMARY")
print("=" * 60)
print(f"  Passed:   {results['passed']}")
print(f"  Failed:   {results['failed']}")
print(f"  Deferred: {results['deferred']} (require live infrastructure)")
print(f"  Total:    {results['passed'] + results['failed'] + results['deferred']}")
print()

if results["failed"] == 0:
    print("✓ ALL NON-DEFERRED ASSERTIONS PASS.")
    print("  Phase 1 is complete. You may proceed to Phase 2.")
    if results["deferred"] > 0:
        print(f"  ({results['deferred']} assertions deferred to deployment — run again with live infra.)")
else:
    print("✗ SOME ASSERTIONS FAILED. Resolve before proceeding to Phase 2.")
    print()
    print("  Failed assertions:")
    for d in results["details"]:
        if d["status"] == "FAIL":
            print(f"    [{d['id']}] {d['desc']}")

## What just changed

You have run the full Phase 1 acceptance suite. Every non-deferred assertion
that passes is a contract honored — a piece of the architecture that works as
specified.

If all assertions pass: Phase 1 is complete. You have a working agentic AI
system that is registry-governed, FIBO-shaped, persona-scoped, and regulatorily
compliant. The next phase (optional, self-paced) extends this to the Wealth
Advisor persona with AgentCore Memory, JWT auth, and three additional agents.

If any assertions fail: the failure message tells you exactly what to fix.
Resolve the failures, re-run this notebook, and confirm all pass before
proceeding.